# Temporal-v3: независимая оценка без обучения

Не нажимай Run all. Выполняй по одной ячейке. После freeze и seal остановись и пришли вывод, прежде чем открывать набор. Старый snapshot holdout не открываем. Старый pilot notebook не запускаем повторно.

Новые 1998 IC-каскадов на том же Facebook-графе; не проверка переноса на другой граф. Frozen seeds 7026/7027/7028, beta=0.5. После открытия настройки не менять и набор не заменять. Время выполнения заранее не гарантируется. Рекомендуется новый runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, importlib, json
from pathlib import Path
REPO = Path(os.environ.get('DIFFUSION_COLAB_REPO', '/content/diffusion-sources'))
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/1habibi/diffusion-sources-localization-', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'networkx', 'numpy', 'scipy', 'matplotlib', 'PyYAML', 'scikit-learn', 'streamlit', 'tqdm', 'torch-geometric', 'pytest'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO), '--no-deps'], check=True)
sys.path.insert(0, str(REPO / 'src'))
# Remove cached package modules and this notebook's stale imported bindings.
for name in list(sys.modules):
    if name == 'diffusion_sources' or name.startswith('diffusion_sources.'):
        del sys.modules[name]
for name in ('PATHS', 'freeze_inputs', 'generate_and_seal', 'open_evaluation', 'evaluate_seed', 'save_summary',
             'REPORT_7026', 'REPORT_7027', 'REPORT_7028'):
    globals().pop(name, None)
importlib.invalidate_caches()
ARTIFACTS = importlib.import_module('diffusion_sources.temporal_independent_artifacts')
GENERATION = importlib.import_module('diffusion_sources.temporal_independent_generation')
INFERENCE = importlib.import_module('diffusion_sources.temporal_independent_inference')
SUMMARY = importlib.import_module('diffusion_sources.temporal_independent_summary')
assert REPO.resolve() in Path(ARTIFACTS.__file__).resolve().parents, 'Imported a different checkout'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Kernel Python:', sys.executable)
print('Project import:', getattr(ARTIFACTS, '__file__', 'module'))
print('PyTorch:', torch.__version__, 'device:', DEVICE)
print('Setup только загрузил код; freeze/seal/open/inference не запускались.')


## 2. Freeze — сверка реальных артефактов

Git не содержит data/raw/facebook_combined.txt.gz. Ячейка ниже получает только этот исходный граф из фиксированного SNAP-источника, если файла нет; существующий файл не заменяется. При повторе после freeze проверяется прежний SHA256. Если загрузка недоступна, положи свою исходную копию по выведенному пути и пришли ошибку; другой граф не подставлять.

Если snapshot metadata отсутствует, остановись и пришли ошибку: нужно найти её настоящий путь. Не создавай замену и не меняй seed. Смена кода/зависимостей после freeze останавливает запуск; не обновляй проект между этапами.

In [ ]:
ROOT = Path('/content/drive/MyDrive/diffusion-sources')
RUN_ROOT = ROOT / 'reports/runs/facebook_main_v2'
PATHS = ARTIFACTS.IndependentPaths(
    repo=REPO, reference=ROOT / 'data/facebook_main',
    snapshot_holdout=ROOT / 'data/facebook_snapshot_final_holdout',
    pilot=ROOT / 'reports/runs/temporal_v3_paired_pilot/v1',
    baseline=ROOT / 'reports/runs/temporal_v3_early_baseline/v1',
    runs={7026: RUN_ROOT / 's1b/seed_7026',
          7027: RUN_ROOT / 'frozen_candidate/seed_7027',
          7028: RUN_ROOT / 'frozen_candidate/seed_7028'},
    data=ROOT / 'data/generated/facebook_temporal_v3_independent_holdout',
    reports=ROOT / 'reports/runs/temporal_v3_independent_evaluation/v1',
    generation_config=REPO / 'configs/facebook_temporal_v3_independent_holdout.yaml')
freeze_inputs = ARTIFACTS.freeze_inputs
generate_and_seal = GENERATION.generate_and_seal
open_evaluation = INFERENCE.open_evaluation
evaluate_seed = INFERENCE.evaluate_seed
save_summary = SUMMARY.save_summary
from diffusion_sources.download import FACEBOOK_URL, download_file
RAW_GRAPH = REPO / 'data/raw/facebook_combined.txt.gz'
print('Required raw graph:', RAW_GRAPH)
if not RAW_GRAPH.is_file():
    raw_sha = None
    prior_manifest = PATHS.reports / 'freeze/manifest.json'
    if prior_manifest.is_file():
        raw_sha = json.loads(prior_manifest.read_text(encoding='utf-8'))['identity']['raw_graph']
    print('Provisioning raw graph only:', FACEBOOK_URL, flush=True)
    try:
        download_file(FACEBOOK_URL, RAW_GRAPH, expected_sha256=raw_sha)
    except (OSError, ValueError) as exc:
        raise RuntimeError(f'Не удалось восстановить {RAW_GRAPH}. Загрузи свою исходную копию по этому пути; не меняй граф/seed.') from exc
    if not RAW_GRAPH.is_file():
        raise RuntimeError(f'Нужен исходный файл {RAW_GRAPH}; пришли ошибку перед freeze.')
print('Reference:', PATHS.reference, '\\nSnapshot metadata:', PATHS.snapshot_holdout)
print('New data:', PATHS.data, '\\nReports:', PATHS.reports)
print('Config:', PATHS.generation_config, 'dataset seed: 4007026; n: 1998; beta: 0.5')
FROZEN = freeze_inputs(PATHS)
print(json.dumps(FROZEN, indent=2))
print('STOP: пришли вывод freeze до следующего этапа.')


## 3. Генерация один раз и seal

Только генерация и проверка seeds/заголовков; качество не вычисляется. Обрыв оставляет артефакты: не удаляй их и не перегенерируй набор. После выполнения остановись и пришли вывод.

In [ ]:
SEALED = generate_and_seal(PATHS)
print(json.dumps(SEALED, indent=2))
print('Seal:', PATHS.reports / 'seal/manifest.json')
print('STOP: набор ещё закрыт. Пришли вывод; ячейку 4 пока НЕ запускать.')


## 4. Открытие и seed 7026 — только после отдельного согласования

Ввод ниже сохраняет необратимый маркер начала оценки до чтения меток. Отрицательный результат тоже сохраняем; не подбираем beta и не выбираем лучший seed.

In [ ]:
answer = input('Введи OPEN_INDEPENDENT_HOLDOUT для независимой оценки без подбора: ').strip()
open_evaluation(PATHS, answer)
REPORT_7026 = evaluate_seed(PATHS, 7026, torch.device(DEVICE))


## 5. Frozen seed 7027

Тот же набор, ранние masks и beta. Только inference.

In [ ]:
REPORT_7027 = evaluate_seed(PATHS, 7027, torch.device(DEVICE))


## 6. Frozen seed 7028

Тот же набор и протокол; все три seeds обязательны независимо от качества.

In [ ]:
REPORT_7028 = evaluate_seed(PATHS, 7028, torch.device(DEVICE))


## 7. Сводка из сохранённых отчётов

1998 парных каскадов, не 5994 независимых примера. CI условен на этом графе/протоколе и трёх checkpoints; разрезы описательные. При провале primary/secondary не открывать другой набор ради успешного результата.

In [ ]:
RESULT = save_summary(PATHS)
print(json.dumps(RESULT, indent=2))
print('Summary:', PATHS.reports / 'summary/payload.json')
